In [ ]:
# Cell: 01 - Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell: 02 - Configure Pandas Display Options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

In [ ]:
# Cell: 03 - Configure Matplotlib and Seaborn
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Cell: 04 - Load Dataset
# Loading exam score prediction dataset from Kaggle
df = pd.read_csv('data.csv')  # Replace with actual file path
print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')

In [ ]:
# Cell: 05 - Display First Few Rows
df.head(10)

In [ ]:
# Cell: 06 - Display Dataset Info
df.info()

In [ ]:
# Cell: 07 - Display Statistical Summary
df.describe()

In [ ]:
# Cell: 08 - Check Dataset Shape and Columns
print(f'Total Rows: {df.shape[0]}')
print(f'Total Columns: {df.shape[1]}')
print(f'\nColumn Names:')
print(df.columns.tolist())

In [ ]:
# Cell: 09 - Check for Missing Values
missing_values = df.isnull().sum()
print('Missing Values:')
print(missing_values[missing_values > 0])
print(f'\nTotal Missing: {df.isnull().sum().sum()}')

In [ ]:
# Cell: 10 - Check Data Types
print('Data Types:')
print(df.dtypes)

In [ ]:
# Cell: 11 - Check for Duplicate Rows
duplicates = df.duplicated().sum()
print(f'Number of Duplicate Rows: {duplicates}')
if duplicates > 0:
    print('Removing duplicates...')
    df = df.drop_duplicates()
    print(f'New shape: {df.shape}')

In [ ]:
# Cell: 12 - Distribution of Target Variable
# Assuming the target variable is the last column (Exam Score)
target_col = df.columns[-1]
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x=target_col, bins=30, kde=True)
plt.title(f'Distribution of {target_col}')
plt.xlabel(target_col)
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Cell: 13 - Statistical Summary of Target Variable
print(f'Target Variable ({target_col}) Statistics:')
print(df[target_col].describe())

In [ ]:
# Cell: 14 - Correlation Matrix
correlation_matrix = df.corr()
print(correlation_matrix)

In [ ]:
# Cell: 15 - Correlation Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Cell: 16 - Correlation with Target Variable
target_correlation = df.corr()[target_col].sort_values(ascending=False)
print(f'Correlation with {target_col}:')
print(target_correlation)

In [ ]:
# Cell: 17 - Visualize Top Correlations
top_features = target_correlation[1:6]
plt.figure(figsize=(10, 6))
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title(f'Top 5 Features Correlated with {target_col}')
plt.xlabel('Correlation Coefficient')
plt.show()

In [ ]:
# Cell: 18 - Pairplot of Top Features
top_features_cols = top_features.index[1:].tolist() + [target_col]
sns.pairplot(df[top_features_cols], diag_kind='kde')
plt.show()

In [ ]:
# Cell: 19 - Check for Outliers (Numerical Columns)
numerical_cols = df.select_dtypes(include=[np.number]).columns
print('Numerical Columns:')
print(numerical_cols.tolist())

In [ ]:
# Cell: 20 - Box Plot for Outlier Detection
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()
for idx, col in enumerate(numerical_cols[:4]):
    sns.boxplot(data=df, y=col, ax=axes[idx])
    axes[idx].set_title(f'Box Plot: {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Cell: 21 - Detect Outliers using IQR Method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers

print('Detecting outliers...')
for col in numerical_cols:
    outliers = detect_outliers_iqr(df, col)
    print(f'{col}: {len(outliers)} outliers')

In [ ]:
# Cell: 22 - Handle Missing Values
print('Handling missing values...')
# Fill numerical columns with median
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Fill categorical columns with mode
for col in df.select_dtypes(include=['object']).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print('Missing values handled!')
print(f'Total missing values: {df.isnull().sum().sum()}')

In [ ]:
# Cell: 23 - Feature Engineering
# Create new features if needed
print('Original features:', df.shape[1])
# Example: Create interaction features
# df['feature_interaction'] = df['feature1'] * df['feature2']
print('Features after engineering:', df.shape[1])

In [ ]:
# Cell: 24 - Separate Features and Target
X = df.drop(columns=[target_col])
y = df[target_col]
print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')

In [ ]:
# Cell: 25 - Identify Categorical and Numerical Features
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
print(f'Categorical Features ({len(categorical_features)}): {categorical_features}')
print(f'Numerical Features ({len(numerical_features)}): {numerical_features}')

In [ ]:
# Cell: 26 - Encode Categorical Variables
from sklearn.preprocessing import LabelEncoder
le_dict = {}
X_encoded = X.copy()
for col in categorical_features:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    le_dict[col] = le
    print(f'Encoded {col}')

In [ ]:
# Cell: 27 - Display Encoded Data
print('Encoded Features:')
print(X_encoded.head())

In [ ]:
# Cell: 28 - Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_scaled = pd.DataFrame(X_scaled, columns=X_encoded.columns)
print('Features Scaled Successfully!')
print(X_scaled.head())

In [ ]:
# Cell: 29 - Check Scaled Data Statistics
print('Scaled Data Statistics:')
print(X_scaled.describe())

In [ ]:
# Cell: 30 - Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

In [ ]:
# Cell: 31 - Check Train-Test Data Distribution
print(f'Target variable distribution in training set:')
print(y_train.describe())
print(f'\nTarget variable distribution in test set:')
print(y_test.describe())

In [ ]:
# Cell: 32 - Import ML Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
print('Models imported successfully!')

In [ ]:
# Cell: 33 - Initialize Models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsRegressor(n_neighbors=5),
    'Support Vector Regression': SVR(kernel='rbf')
}
print(f'Total models: {len(models)}')

In [ ]:
# Cell: 34 - Train Linear Regression Model
lr_model = models['Linear Regression']
lr_model.fit(X_train, y_train)
print('Linear Regression trained!')

In [ ]:
# Cell: 35 - Train Decision Tree Model
dt_model = models['Decision Tree']
dt_model.fit(X_train, y_train)
print('Decision Tree trained!')

In [ ]:
# Cell: 36 - Train Random Forest Model
rf_model = models['Random Forest']
rf_model.fit(X_train, y_train)
print('Random Forest trained!')

In [ ]:
# Cell: 38 - Train K-Nearest Neighbors Model
knn_model = models['K-Nearest Neighbors']
knn_model.fit(X_train, y_train)
print('K-Nearest Neighbors trained!')

In [ ]:
# Cell: 40 - Import Evaluation Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math
print('Metrics imported successfully!')

In [ ]:
# Cell: 41 - Function to Evaluate Models
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    train_rmse = math.sqrt(train_mse)
    test_rmse = math.sqrt(test_mse)
    
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    print(f'\n{model_name} Performance:')
    print(f'Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}')
    print(f'Train MAE: {train_mae:.4f}, Test MAE: {test_mae:.4f}')
    print(f'Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}')
    
    return {
        'Model': model_name,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train R²': train_r2,
        'Test R²': test_r2
    }

print('Evaluation function created!')